# Task 3 — Predictive Grid Management for Heat Pump Networks
**Goal:** Predict monthly average `x2` (grid load) per device for May–Oct 2025  
**Metric:** MAE (lower = better)  
**Train:** Oct 2024 – Apr 2025 | **Val:** May–Jun 2025 | **Test:** Jul–Oct 2025

## 1. Setup

In [ ]:
!pip install lightgbm pandas numpy scikit-learn pyarrow -q

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

FORECAST_MONTHS = [5, 6, 7, 8, 9, 10]
FORECAST_YEAR   = 2025
RANDOM_STATE    = 42

## 2. Config — local or Colab

In [ ]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
ENV = 'colab'   # 'local' or 'colab'

LOCAL_DATA_DIR = '../solution_task3/data'          # path when running locally
COLAB_DATA_DIR = '/content/drive/MyDrive/task3_data'  # path on Google Drive

# Fraction of rows to sample per chunk (1.0 = all rows, 0.1 = 10% = ~1GB)
# Lower this if you run out of RAM. 0.2 works fine for LightGBM.
SUBSAMPLE_FRAC = 0.2

CHUNK_SIZE = 500_000   # rows read at once — tune down if RAM is tight
# ──────────────────────────────────────────────────────────────────────────────

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = COLAB_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR

print(f'ENV={ENV}  DATA_DIR={DATA_DIR}  SUBSAMPLE_FRAC={SUBSAMPLE_FRAC}')

## 3. Load data in chunks with subsampling
10 GB file — we read it in chunks and keep only a random fraction of rows per chunk.  
After chunking we aggregate straight to monthly level so RAM usage stays low.

In [ ]:
def load_chunked(path, chunk_size, subsample_frac, seed=42):
    """
    Read a large CSV in chunks, randomly subsample each chunk,
    and return the concatenated result.
    """
    rng = np.random.default_rng(seed)
    chunks = []
    reader = pd.read_csv(
        path,
        parse_dates=['Timedate'],
        chunksize=chunk_size,
    )
    for i, chunk in enumerate(reader):
        if subsample_frac < 1.0:
            mask = rng.random(len(chunk)) < subsample_frac
            chunk = chunk[mask]
        chunks.append(chunk)
        if (i + 1) % 10 == 0:
            print(f'  chunk {i+1} done, rows so far: {sum(len(c) for c in chunks):,}')

    df = pd.concat(chunks, ignore_index=True)
    return df


print(f'Loading data.csv  (chunk={CHUNK_SIZE:,}, subsample={SUBSAMPLE_FRAC}) ...')
df = load_chunked(
    os.path.join(DATA_DIR, 'data.csv'),
    chunk_size=CHUNK_SIZE,
    subsample_frac=SUBSAMPLE_FRAC,
)

devices = pd.read_csv(os.path.join(DATA_DIR, 'devices.csv'))
df = df.merge(devices, on='deviceId', how='left')

print(f'\nLoaded {len(df):,} rows  |  {df["deviceId"].nunique()} devices')
print(f'Date range: {df["Timedate"].min()} → {df["Timedate"].max()}')
print(f'RAM usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

## 4. EDA

In [ ]:
print('--- missing values (%) ---')
print((df.isnull().mean() * 100).round(2))

In [ ]:
df['year']  = df['Timedate'].dt.year
df['month'] = df['Timedate'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['x2'].hist(bins=60, ax=axes[0])
axes[0].set_title('x2 distribution')

monthly_mean = df.groupby(['year', 'month'])['x2'].mean().reset_index()
monthly_mean['date'] = pd.to_datetime(monthly_mean[['year', 'month']].assign(day=1))
monthly_mean.set_index('date')['x2'].plot(ax=axes[1], marker='o')
axes[1].set_title('Monthly mean x2 (fleet-wide)')

plt.tight_layout()
plt.show()

In [ ]:
temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
corr = df[temp_cols + ['x1', 'x2']].corr()['x2'].drop('x2').sort_values()
corr.plot(kind='barh', figsize=(8, 5), title='Feature correlation with x2')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Feature Engineering & monthly aggregation

In [ ]:
def build_features(df):
    df = df.copy()
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    if 'deviceType' in df.columns:
        le = LabelEncoder()
        df['deviceType_enc'] = le.fit_transform(df['deviceType'].astype(str))
    return df


def aggregate_monthly(df):
    temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
    agg = {c: 'mean' for c in temp_cols}
    for col in ['x1', 'x2', 'month_sin', 'month_cos']:
        if col in df.columns:
            agg[col] = 'mean'
    for col in ['latitude', 'longitude', 'deviceType_enc', 'x3']:
        if col in df.columns:
            agg[col] = 'first'
    return df.groupby(['deviceId', 'year', 'month']).agg(agg).reset_index()


df      = build_features(df)
monthly = aggregate_monthly(df)

# Free raw data from RAM
del df

print(f'Monthly rows: {len(monthly):,}')
monthly.head()

In [ ]:
# Lag features on monthly aggregates (fast — monthly table is tiny)
monthly = monthly.sort_values(['deviceId', 'year', 'month']).reset_index(drop=True)

for lag in [1, 2, 3]:
    monthly[f'x2_lag{lag}'] = monthly.groupby('deviceId')['x2'].shift(lag)

monthly['x2_roll3'] = monthly.groupby('deviceId')['x2'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

monthly[['deviceId', 'year', 'month', 'x2', 'x2_lag1', 'x2_lag2', 'x2_roll3']].head(10)

## 6. Train / Validation split

In [ ]:
FEATURE_COLS = [
    'month_sin', 'month_cos',
    *[f't{i}' for i in range(1, 14) if f't{i}' in monthly.columns],
    'x1', 'x3',
    'latitude', 'longitude', 'deviceType_enc',
    'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in monthly.columns]
print('Features:', FEATURE_COLS)

train_df = monthly[monthly['x2'].notna()].copy()
val_df   = monthly[
    (monthly['year'] == 2025) & (monthly['month'].isin([5, 6])) & monthly['x2'].notna()
].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df['x2']
X_val,   y_val   = val_df[FEATURE_COLS],   val_df['x2']

print(f'Train: {len(X_train)}  |  Val: {len(X_val)}')

## 7. Train LightGBM

In [ ]:
import lightgbm as lgb

model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ]
)

val_preds = model.predict(X_val)
val_mae   = mean_absolute_error(y_val, val_preds)
print(f'\nValidation MAE: {val_mae:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values().plot(
    kind='barh', ax=axes[0], title='Feature importance'
)

axes[1].scatter(y_val, val_preds, alpha=0.4, s=10)
lims = [min(float(y_val.min()), val_preds.min()), max(float(y_val.max()), val_preds.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_xlabel('Actual x2')
axes[1].set_ylabel('Predicted x2')
axes[1].set_title(f'Val MAE = {val_mae:.4f}')

plt.tight_layout()
plt.show()

## 8. Retrain on all labeled data & generate submission

In [ ]:
all_labeled = monthly[monthly['x2'].notna()].copy()
model_final = LGBMRegressor(**model.get_params())
model_final.set_params(n_estimators=model.best_iteration_ + 50)
model_final.fit(all_labeled[FEATURE_COLS], all_labeled['x2'])
print(f'Retrained on {len(all_labeled)} samples')

In [ ]:
def generate_submission(model, monthly, feature_cols):
    rows = []
    for device_id, dev in monthly.groupby('deviceId'):
        dev = dev.sort_values(['year', 'month'])
        static_mean = dev[
            [c for c in feature_cols if c not in
             ('month_sin', 'month_cos', 'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3')]
        ].mean()
        last_x2 = dev['x2'].dropna().tolist()

        for month in FORECAST_MONTHS:
            feat = static_mean.copy()
            feat['month_sin'] = np.sin(2 * np.pi * month / 12)
            feat['month_cos'] = np.cos(2 * np.pi * month / 12)
            if 'x2_lag1' in feature_cols:
                feat['x2_lag1'] = last_x2[-1] if len(last_x2) >= 1 else np.nan
            if 'x2_lag2' in feature_cols:
                feat['x2_lag2'] = last_x2[-2] if len(last_x2) >= 2 else np.nan
            if 'x2_lag3' in feature_cols:
                feat['x2_lag3'] = last_x2[-3] if len(last_x2) >= 3 else np.nan
            if 'x2_roll3' in feature_cols:
                feat['x2_roll3'] = np.mean(last_x2[-3:]) if last_x2 else np.nan

            pred = float(np.clip(model.predict(pd.DataFrame([feat[feature_cols]]))[0], 0.0, 1.0))
            rows.append((device_id, FORECAST_YEAR, month, pred))
            last_x2.append(pred)

    return pd.DataFrame(rows, columns=['deviceId', 'year', 'month', 'prediction'])


submission = generate_submission(model_final, monthly, FEATURE_COLS)
print(f'Submission rows: {len(submission)}')
submission.head(12)

## 9. Save submission & model

In [ ]:
OUT_DIR = os.path.join(DATA_DIR, 'out') if ENV == 'colab' else 'data/out'
os.makedirs(OUT_DIR, exist_ok=True)

submission.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)
print(f'Saved submission → {OUT_DIR}/submission.csv')

with open(os.path.join(OUT_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump({'model': model_final, 'features': FEATURE_COLS}, f)
print(f'Saved model      → {OUT_DIR}/model.pkl')

# On Colab: also download locally
if ENV == 'colab':
    from google.colab import files
    submission.to_csv('submission.csv', index=False)
    files.download('submission.csv')
    files.download(os.path.join(OUT_DIR, 'model.pkl'))

## 10. Sanity check

In [ ]:
print(submission.groupby('month')['prediction'].agg(['mean', 'std', 'min', 'max']).round(4))
assert submission[['deviceId', 'year', 'month']].duplicated().sum() == 0, 'Duplicate rows!'
assert submission['prediction'].between(0, 1).all(), 'Predictions out of [0,1]!'
print('\nAll checks passed.')